::: {.sb-downloads}
[Download this notebook](../downloads/solutions-house-prices.ipynb){download="solutions-house-prices.ipynb"}  
Data: [house_prices.csv](../downloads/data/house_prices.csv){download="house_prices.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

::: {.sb-lesson-links}
[← Back to the exercises](../exercises/house-prices.ipynb)
:::

::: {.callout-tip}
Try the exercises on your own first: you learn much more by searching than by reading a solution.
:::

PCA on tabular data: can a few components summarise the houses of a city? And what do these components mean?

## The dataset {#ex-the-dataset}
`../data/house_prices.csv` is the Ames Housing dataset (D. De Cock, 2011): 1,460 houses sold in Ames (Iowa, USA) between 2006 and 2010, described by 79 variables (surfaces, quality grades, year of construction, garage, basement...), plus an `Id` and the sale price `SalePrice`, in dollars. The surfaces are in square feet.

1. Import the libraries and load the dataset (given).

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

houses = pd.read_csv("../data/house_prices.csv")
houses.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Preparing the data {#ex-preparing-the-data}

2. PCA only works with numbers. Keep the numeric columns only (`select_dtypes`), remove `Id` (a row number tells nothing about a house) and put `SalePrice` aside in a variable `price`: we want to describe the houses, not their price. How many variables are left?

In [ ]:
price = houses["SalePrice"]
X = houses.select_dtypes(include="number").drop(columns=["Id", "SalePrice"])
print(X.shape)

(1460, 36)


**36 numeric variables are left for the 1,460 houses.**

3. Which columns have missing values, and how many? PCA cannot handle missing values: fill them with the median of each column (`SimpleImputer`).

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `X.isna().sum()` counts the missing values of each column. `SimpleImputer(strategy="median").fit_transform(X)` returns a numpy array: put it back in a DataFrame with the same column names.
:::

In [ ]:
missing = X.isna().sum()
print(missing[missing > 0])

X_filled = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X), columns=X.columns)
print("Missing values left:", X_filled.isna().sum().sum())

LotFrontage    259
MasVnrArea       8
GarageYrBlt     81
dtype: int64
Missing values left: 0


**Three columns have missing values: `LotFrontage` (259, the length of street in front of the lot), `GarageYrBlt` (81, the houses without a garage) and `MasVnrArea` (8). Dropping these rows would lose 339 houses (23%), so we fill them with the median.**

4. Look at the variance of a few columns (`LotArea`, `GrLivArea`, `OverallQual`, `YrSold`). Why must we standardise the data before the PCA? Standardise it.

In [ ]:
print(X_filled[["LotArea", "GrLivArea", "OverallQual", "YrSold"]].var().round(1))

X_scaled = StandardScaler().fit_transform(X_filled)

LotArea        99625649.7
GrLivArea        276129.6
OverallQual           1.9
YrSold                1.8
dtype: float64


**The variances are completely different: about 99.6 million for `LotArea` (square feet), 276,000 for `GrLivArea`, 1.9 for `OverallQual` (a grade from 1 to 10) and 1.8 for `YrSold`. Without scaling, the first components would only describe the lot area and the living area, just because of their units. After standardisation, every variable has a variance of 1.**

## How many components? {#ex-how-many-components}

5. Fit a PCA with all the components on the standardised data. Plot the explained variance ratio of each component and the cumulative ratio. How much does PC1 keep? How many components do you need for 80% of the variance? For 90%?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `np.argmax(cumulative >= 0.8) + 1` gives the first number of components that reaches 80%.
:::

In [ ]:
pca = PCA().fit(X_scaled)
ratio = pca.explained_variance_ratio_
cumulative = ratio.cumsum()
print(f"PC1: {ratio[0]:.1%}, PC2: {ratio[1]:.1%}, PC3: {ratio[2]:.1%}")
for target in [0.8, 0.9]:
    print(f"{target:.0%} of the variance: {np.argmax(cumulative >= target) + 1} components")

k = np.arange(1, len(ratio) + 1)
fig = go.Figure()
fig.add_trace(go.Bar(x=k, y=ratio, name="ratio"))
fig.add_trace(go.Scatter(x=k, y=cumulative, mode="lines+markers", name="cumulative"))
for target in [0.8, 0.9]:
    fig.add_hline(y=target, line_dash="dot", line_color="#999999", annotation_text=f"{target:.0%}")
fig.update_layout(title="House prices: explained variance", xaxis_title="component", yaxis_title="share of the variance")
fig.show()

PC1: 19.8%, PC2: 8.9%, PC3: 7.1%
80% of the variance: 18 components
90% of the variance: 22 components


**PC1 keeps 19.8% of the variance, PC2 8.9% and PC3 7.1%. We need 18 components for 80% and 22 for 90%: much less compression than with images. The 36 variables are only partly redundant: many of them (pool area, porch areas, month of sale...) carry their own, independent information.**

6. Check your answer with `PCA(n_components=0.8)` and `PCA(n_components=0.9)`: how many components does scikit-learn keep?

In [ ]:
for share in [0.8, 0.9]:
    p = PCA(n_components=share).fit(X_scaled)
    print(f"PCA(n_components={share}): {p.n_components_} components, {p.explained_variance_ratio_.sum():.1%} of the variance")

PCA(n_components=0.8): 18 components, 82.2% of the variance
PCA(n_components=0.9): 22 components, 90.0% of the variance


**The same: 18 components (82.2% of the variance) and 22 components (90.0%).**

## What do the components mean? {#ex-what-do-the-components-mean}

7. Put the loadings of the first 3 components in a DataFrame (one row per variable). For each component, display the 7 variables with the largest weights in absolute value. Try to give a name to PC1 and PC2.

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `pca.components_[:3].T` has one row per variable. `series.sort_values(key=abs, ascending=False)` sorts by absolute value.
:::

In [ ]:
loadings = pd.DataFrame(pca.components_[:3].T, index=X.columns, columns=["PC1", "PC2", "PC3"])
for pc in loadings.columns:
    print(pc, loadings[pc].sort_values(key=abs, ascending=False).head(7).round(2).to_dict())

PC1 {'OverallQual': 0.3, 'GrLivArea': 0.3, 'GarageCars': 0.28, 'GarageArea': 0.28, 'FullBath': 0.26, 'TotalBsmtSF': 0.26, '1stFlrSF': 0.26}
PC2 {'2ndFlrSF': 0.43, 'BedroomAbvGr': 0.36, 'TotRmsAbvGrd': 0.33, 'BsmtFinSF1': -0.29, 'BsmtFullBath': -0.29, 'GrLivArea': 0.25, 'HalfBath': 0.22}
PC3 {'GarageYrBlt': 0.36, 'YearBuilt': 0.32, 'LotArea': -0.28, 'LotFrontage': -0.27, 'YearRemodAdd': 0.27, '1stFlrSF': -0.26, 'MSSubClass': 0.23}


**PC1: all its large weights are positive and about the same size: `OverallQual` (0.30), `GrLivArea` (0.30), `GarageCars` and `GarageArea` (0.28), `FullBath`, `TotalBsmtSF`, `1stFlrSF` (0.26). A house with a high PC1 is big and of good quality on every point: PC1 is a "size and quality" score.**

**PC2 opposes the upper floor (`2ndFlrSF` 0.43, `BedroomAbvGr` 0.36, `TotRmsAbvGrd` 0.33) to the finished basement (`BsmtFinSF1` −0.29, `BsmtFullBath` −0.29): it describes how the house is organised, upstairs or in the basement, rather than its size.**

**PC3 opposes recent houses (`GarageYrBlt` 0.36, `YearBuilt` 0.32) to large lots (`LotArea` −0.28, `LotFrontage` −0.27): recent houses tend to be built on smaller lots.**

## Drawing the houses {#ex-drawing-the-houses}

8. Draw the houses on PC1 and PC2, coloured by `SalePrice`. What do you notice? Compute the correlation between the price and PC1, and between the price and PC2.

In [ ]:
Z = pca.transform(X_scaled)
fig = px.scatter(x=Z[:, 0], y=Z[:, 1], color=price, color_continuous_scale="Viridis",
                 labels={"x": f"PC1 ({ratio[0]:.1%})", "y": f"PC2 ({ratio[1]:.1%})", "color": "SalePrice ($)"},
                 title="1,460 houses on their first two principal components", opacity=0.7)
fig.show()

print(f"Correlation price / PC1: {np.corrcoef(Z[:, 0], price)[0, 1]:.2f}")
print(f"Correlation price / PC2: {np.corrcoef(Z[:, 1], price)[0, 1]:.2f}")

Correlation price / PC1: 0.86
Correlation price / PC2: -0.05


**The price grows from left to right: the expensive houses all have a high PC1. The correlation between the price and PC1 is 0.86, although PCA never saw the price: the "size and quality" score found without the price is a very good summary of what makes a house expensive. PC2 is almost unrelated to the price (correlation −0.05): it describes how the house is organised, not its value. A few points far on the right are very large houses (outliers).**

## Optional: a model on the components {#ex-optional-a-model-on-the-components}

9. Split the data (20% for the test set, `random_state=0`, keeping the raw numeric `X` and `price`). Build 3 pipelines (imputer → scaler → [PCA] → `RandomForestRegressor(random_state=0)`): without PCA, with 3 components, and with 80% of the variance. Compare their $R^2$ on the test set. What do you conclude?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), PCA(n_components=3), RandomForestRegressor(random_state=0))`. The pipeline fits the imputer, the scaler and the PCA on the training set only.
:::

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, price, test_size=0.2, random_state=0)

for name, pca_step in [("36 original variables", []), ("3 components", [PCA(n_components=3)]),
                       ("80% of the variance", [PCA(n_components=0.8)])]:
    pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), *pca_step, RandomForestRegressor(random_state=0))
    pipe.fit(X_train, y_train)
    print(f"{name:<22} {pipe[-1].n_features_in_:>2} features   test R2 = {pipe.score(X_test, y_test):.3f}")

36 original variables  36 features   test R2 = 0.834


3 components            3 features   test R2 = 0.841


80% of the variance    17 features   test R2 = 0.792


**The forest on the 36 variables reaches $R^2$ = 0.834 on the test set; on only 3 components, 0.841: as good, with 12 times fewer features. With 80% of the variance (17 components on the training set) it drops to 0.792. More variance kept is not always better for a model: PCA keeps the directions where the houses differ the most, not the directions that predict the price. PC1 happens to be very linked to the price; the next components add mostly variations that do not matter for the price. The number of components of a model should be chosen by cross-validation, not by a share of variance.**